In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import csv, time

BASE_URL = "https://propertyguys.com/search/ca/on/Hamilton"

def scrape_listings(output_csv="hamilton_listings.csv"):
    options = Options()
    options.add_argument("--window-size=400,900")  # mobile layout
    # options.add_argument("--headless")
    driver = webdriver.Chrome(service=Service(), options=options)

    driver.get(BASE_URL)

    WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img.listing-photo"))
    )

    results = []
    page_num = 1

    while True:
        print(f"📄 Scraping page {page_num}...")
        listing_photos = driver.find_elements(By.CSS_SELECTOR, "img.listing-photo")
        print(f"Found {len(listing_photos)} listings on this page")

        for i in range(len(listing_photos)):
            try:
                photo = driver.find_elements(By.CSS_SELECTOR, "img.listing-photo")[i]
                driver.execute_script("arguments[0].click();", photo)

                WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "h2.street"))
                )

                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)

                try:
                    address = driver.find_element(By.CSS_SELECTOR, "h2.street").text.strip()
                except:
                    address = ""

                try:
                    price_elem = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.price-container.mobile h3"))
                    )
                    driver.execute_script("arguments[0].scrollIntoView(true);", price_elem)
                    price = price_elem.text.strip()
                except:
                    price = ""

                try:
                    phone_elem = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.mobile-contact-card a.phone"))
                    )
                    driver.execute_script("arguments[0].scrollIntoView(true);", phone_elem)
                    phone = phone_elem.text.strip()
                except:
                    phone = ""

                url = driver.current_url

                results.append({
                    "listing_url": url,
                    "address": address,
                    "phone": phone,
                    "price": price
                })
                print(f"✅ Scraped: {address} | {price} | {phone}")

                driver.back()
                WebDriverWait(driver, 20).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img.listing-photo"))
                )
                time.sleep(1)

            except Exception as e:
                print(f"⚠️ Skipping listing {i+1}, error: {e}")
                driver.back()
                WebDriverWait(driver, 20).until(
                    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img.listing-photo"))
                )
                time.sleep(1)

        # Try to click next page
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, "button.next-button")
            if "disabled" in next_btn.get_attribute("class"):
                print("🚫 No more pages.")
                break
            driver.execute_script("arguments[0].click();", next_btn)
            page_num += 1
            time.sleep(3)
            WebDriverWait(driver, 20).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "img.listing-photo"))
            )
        except:
            print("🚫 Could not find next button. Stopping.")
            break

    driver.quit()

    # Save CSV
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["listing_url", "address", "phone", "price"])
        writer.writeheader()
        writer.writerows(results)

    print(f"🎉 Done! Saved {len(results)} listings to {output_csv}")

if __name__ == "__main__":
    scrape_listings()


📄 Scraping page 1...
Found 11 listings on this page
✅ Scraped: 16 Birmingham St | $5,000,000 | (416) 873-1347
✅ Scraped: 1401 King St E | $5,000,000 | (416) 873-1347
✅ Scraped: 216 & 218 King St E | $4,000,000 | (416) 873-1347
✅ Scraped: 1063 Barton St E | $4,000,000 | (416) 873-1347
✅ Scraped: 1221-1223 Main st. E | $1,150,000 | (855) 742-4539
✅ Scraped: 206 Wilson St | $1,100,000 | (416) 873-1347
✅ Scraped: 88 Sanford Ave N | $999,000 | (855) 742-4539
✅ Scraped: 315 Garner Rd W | $885,900 | (855) 742-4539
✅ Scraped: 18 Rosedene Ave | $849,000 | (855) 742-4539
✅ Scraped: 561 King St E | $799,000 | (416) 873-1347
✅ Scraped: 50 Emick Dr | $699,000 | (855) 742-4539
📄 Scraping page 2...
Found 11 listings on this page
✅ Scraped: 77 A Chestnut Ave | $624,000 | (855) 742-4539


KeyboardInterrupt: 